# Vehicle Budget Advisor — Model Training (Kaggle)

Ye notebook **`vehicles.csv`** par Machine Learning chalati hai:
1. **Random Forest Regressor** — vehicle ki market price estimate karta hai
2. **K-Nearest Neighbours** — similar vehicles dhoondta hai

**Kaise use karein:**
- Kaggle par notebook kholen → `+ Add Input` se apna `vehicles.csv` upload/dataset attach karein
- `Run All` dabayein → neeche `model.joblib` download karein
- Us file ko apni repo me app ke sath rakh dein (Streamlit app use kar legi)

In [ ]:
import os
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# NOTE: Kaggle par file ka path alag ho sakta hai — apne input folder ke mutabiq badlein
CSV = "vehicles.csv"
df = pd.read_csv(CSV)
df["price_pkr"] = df["price_pkr"].astype(float)
df["seats"] = df["seats"].astype(int)
df["top_speed_kmh"] = df["top_speed_kmh"].astype(int)
print(f"{len(df)} vehicles loaded · categories: {df['category'].nunique()}")
df.head()

## 1) Random Forest — price estimator

In [ ]:
CAT_COLS = ["category", "brand", "fuel_type"]
NUM_COLS = ["seats", "top_speed_kmh"]
X, y = df[CAT_COLS + NUM_COLS], df["price_pkr"]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ("num", StandardScaler(), NUM_COLS),
])
model = Pipeline([("pre", pre),
                  ("rf", RandomForestRegressor(n_estimators=250, random_state=42))])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(Xtr, ytr)
pred = model.predict(Xte)
mae = mean_absolute_error(yte, pred)
print(f"MAE = {mae:,.0f} PKR  (~{(mae / y.mean()) * 100:.1f}% of average price)")

# full-data model -> model.joblib
model.fit(X, y)
joblib.dump(model, "model.joblib")
print("model.joblib saved ✅")

# example prediction
ex = pd.DataFrame([{"category": "Car", "brand": "Honda", "fuel_type": "Petrol",
                    "seats": 5, "top_speed_kmh": 190}])
print(f"Example estimate (Honda-like sedan): {model.predict(ex)[0]:,.0f} PKR")

## 2) K-Nearest Neighbours — similar vehicles
App khud runtime par bhi KNN chala leti hai; ye cell sirf verification ke liye hai.

In [ ]:
feats = df[["seats", "top_speed_kmh", "price_pkr"]].astype(float)
Xs = StandardScaler().fit_transform(feats)
nn = NearestNeighbors(n_neighbors=6).fit(Xs)

i = df.index[df["name"].str.contains("Civic")][0]
_, idx = nn.kneighbors([Xs[i]])
print("Similar to:", df.loc[i, "name"])
df.iloc[idx[0][1:]][["name", "brand", "category", "price_pkr"]]

## 3) Download `model.joblib`
Right panel → Output → `model.joblib` download karein aur repo me app ke sath rakh dein.
Agar file na ho to Streamlit app CSV se khud model train kar legi.